# 🚚 Commercial Fleet OOS Opportunity AI Portal — Test Notebook
This Jupyter Notebook provides end-to-end testing for all core functionalities of the **Carrier Fix** Azure Cloud Application:
1. **Environment Credentials & Azure Connection Check**
2. **Step 1: Scheduled Raw Data Ingestion (SoQL API Date >= 2026-01-01 & ACTIVE status)**
3. **Step 2: Raw Snapshot Upload to Azure Blob Storage (`oos-raw`)**
4. **Step 3: Carrier Data Enrichment & Upload to Azure Blob Storage (`oos-enriched`)**
5. **Step 4: Index Enriched Carrier Documents to Azure AI Search (`poc_oos_enriched`)**
6. **Step 5: Territory Location Search & Filtering Query**
7. **Step 6: Azure OpenAI Custom AI Email Campaign Generation (`gpt-5-mini`)**

## 1. Environment & Azure Credentials Verification
Loads environment variables from `.env` and tests connections to Azure Storage, Azure OpenAI, and Azure AI Search.

In [ ]:
import os, ssl, urllib3
from dotenv import load_dotenv

ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()
load_dotenv()

print('=== ENVIRONMENT CREDENTIALS LOADED ===')
print('AZURE_OPENAI_ENDPOINT:', os.getenv('AZURE_OPENAI_ENDPOINT'))
print('AZURE_OPENAI_DEPLOYMENT:', os.getenv('AZURE_OPENAI_DEPLOYMENT'))
print('AZURE_SEARCH_ENDPOINT:', os.getenv('AZURE_SEARCH_ENDPOINT'))
print('AZURE_SEARCH_INDEX:', os.getenv('AZURE_SEARCH_INDEX'))
print('CONTAINER_OOS_RAW:', os.getenv('CONTAINER_OOS_RAW'))
print('CONTAINER_OOS_ENRICHED:', os.getenv('CONTAINER_OOS_ENRICHED'))

## 2. Step 1: Scheduled Raw Data Fetch (SoQL API Query)
Tests `AzureFunctionIngestionJob` fetching Out-of-Service records starting from `2026-01-01` with `status == ACTIVE` using SoQL query parameters (`$select`, `$where`, `$order`, `$limit`).

In [ ]:
from azure_function_ingestion import AzureFunctionIngestionJob

ingestion_job = AzureFunctionIngestionJob(target_date_start='2026-01-01')
ingest_result = ingestion_job.execute_scheduled_ingestion(limit=20)

raw_records = ingest_result.get('records', [])
print(f"Ingestion Status: {ingest_result.get('status')}")
print(f"Raw Records Fetched: {len(raw_records)}")
if raw_records:
    print('Sample Raw Record:', raw_records[0])

## 3. Step 2: Store Raw Snapshot to Azure Blob Storage (`oos-raw`)
Saves raw snapshot to CSV and uploads it to Azure Blob Storage container `oos-raw` on account `wynstore5505`.

In [ ]:
import pandas as pd
from utils import upload_file_to_blob

raw_df = pd.DataFrame(raw_records)
os.makedirs('data', exist_ok=True)
raw_test_file = 'data/test_raw_snapshot.csv'
raw_df.to_csv(raw_test_file, index=False)

raw_container = os.getenv('CONTAINER_OOS_RAW', 'oos-raw')
blob_raw_url = upload_file_to_blob(raw_test_file, container_name=raw_container, blob_name='test_raw_snapshot.csv')
print('Uploaded Raw Snapshot Blob URL:', blob_raw_url)

## 4. Step 3: Carrier Data Enrichment & Upload to Azure Blob Storage (`oos-enriched`)
Enriches raw carrier profiles using Child Agent (`CarrierEnrichmentChildAgent`) and uploads enriched dataset to container `oos-enriched`.

In [ ]:
from child_agent_enrichment import CarrierEnrichmentChildAgent

child_agent = CarrierEnrichmentChildAgent(use_live_api=False)
enriched_records = []

for rec in raw_records[:10]:
    dot = rec.get('DOT_NUMBER')
    if dot:
        enr = child_agent.enrich_carrier_by_dot(int(dot))
        combined = {**rec, **enr}
        enriched_records.append(combined)

enriched_df = pd.DataFrame(enriched_records)
enriched_test_file = 'data/test_enriched_snapshot.csv'
enriched_df.to_csv(enriched_test_file, index=False)

enriched_container = os.getenv('CONTAINER_OOS_ENRICHED', 'oos-enriched')
blob_enriched_url = upload_file_to_blob(enriched_test_file, container_name=enriched_container, blob_name='test_enriched_snapshot.csv')
print(f"Enriched {len(enriched_records)} records.")
print('Uploaded Enriched Snapshot Blob URL:', blob_enriched_url)

## 5. Step 4: Index Enriched Documents to Azure AI Search (`poc_oos_enriched`)
Indexes carrier documents into Azure AI Search index `poc_oos_enriched`.

In [ ]:
from azure_ai_search_service import AzureAISearchService

search_service = AzureAISearchService()
indexed_docs = search_service.index_enriched_records(enriched_records)
print(f"Indexed {len(indexed_docs)} documents to Azure AI Search index '{search_service.index_name}'.")

## 6. Step 5: Territory Location Search & Filtering Query
Queries Azure AI Search filtering by salesperson territory states (e.g. `['TX', 'FL', 'IL']`), date `>= 2026-01-01`, and `STATUS == 'ACTIVE'`.

In [ ]:
search_results = search_service.search_by_territory_states(
    territory_states=['TX', 'FL', 'IL', 'CA'],
    start_date='2026-01-01',
    status='ACTIVE',
    search_query='*',
    limit=10
)

df_results = pd.DataFrame(search_results)
print(f"Retrieved {len(df_results)} territory search results:")
if not df_results.empty:
    display(df_results[['DOT_NUMBER', 'LEGAL_NAME', 'PHY_CITY', 'PHY_STATE', 'OOS_DATE', 'STATUS']])

## 7. Step 6: Azure OpenAI Custom Sales Email Generation (`gpt-5-mini`)
Invokes `AzureFoundrySalesAgent` to generate a custom outreach email for a targeted OOS carrier company using **Carrier Fix** branding, sales rep details, and territory area.

In [ ]:
from azure_foundry_agent import AzureFoundrySalesAgent

agent = AzureFoundrySalesAgent()

sample_carrier = {
    'DOT_NUMBER': 1438,
    'LEGAL_NAME': 'AUSTIN URETHANE INC',
    'OOS_DATE': '2026-02-15',
    'OOS_REASON': 'Unsatisfactory = Unfit',
    'PHY_CITY': 'Austin',
    'PHY_STATE': 'TX',
    'VEHICLE_TOTAL': 14,
    'DRIVER_TOTAL': 8
}

sample_rep = {
    'name': 'John Doe',
    'title': 'Southern Region Fleet Replacement Specialist',
    'region': 'SOUTH',
    'email': 'john.doe@carrierfix.com',
    'focus_offering': 'Asset Replacement & Rapid Lease Program'
}

email_payload = agent.generate_custom_email_for_carrier(sample_carrier, sample_rep)

print('Company:', email_payload['COMPANY_NAME'])
print('Sales Representative:', email_payload['SALESPERSON'])
print('Territory Area:', email_payload['SALES_REGION'])
print('Subject:', email_payload['SUBJECT'])
print('\n' + '='*60)
print(email_payload['FULL_EMAIL_BODY'])
print('='*60)